# Workshop Progress Recorder

Records the result of every workshop check in the shared progress table, so the Workshop Guide app can show it live.

> **Note:** Participants never edit or open this notebook. The workshop checks load it with `%run ./_progress`;
> the table name `PROGRESS_TABLE` comes from `00_setup`.

## What it provides

- `record(workshop, step, step_type, ok, message)` appends one row for the current user to `PROGRESS_TABLE`
- `_result(ok, message)` prints `PASS — message` or `FAIL — message` in the notebook

In [ ]:
# Table name from 00_setup; fallback when this notebook runs on its own
try:
    PROGRESS_TABLE
except NameError:
    PROGRESS_TABLE = "retailhub_trainer.default.workshop_progress"

# Print "Progress not saved" only once per notebook session, even if this notebook is loaded again
_progress_warned = globals().get("_progress_warned", False)

In [ ]:
# One consistent feedback line for every check
def _result(ok, message):
    """Print 'PASS — message' or 'FAIL — message' and return ok."""
    print(f"{'PASS' if ok else 'FAIL'} — {message}")
    return ok

## Writing a row

One parameterized `INSERT` per check: `user_email` is `current_user()`, `source` is `notebook`,
`hints_used` and `attempts` stay `NULL` (only the app knows them). A failed write never stops the workshop.

In [ ]:
# INSERT with named parameters (:workshop, ...); {table} is filled in at call time
_PROGRESS_INSERT = """
INSERT INTO {table}
  (user_email, workshop, step, step_type, status, message, hints_used, attempts, source, ts)
VALUES
  (current_user(), :workshop, :step, :step_type, :status, :message,
   NULL, NULL, 'notebook', current_timestamp())
"""

In [ ]:
# Append one check result; on failure print one warning and carry on
def record(workshop, step, step_type, ok, message):
    """Append a row to PROGRESS_TABLE for the current user. Never raises."""
    global _progress_warned
    args = {"workshop": workshop, "step": step, "step_type": step_type,
            "status": "pass" if ok else "fail", "message": str(message)[:200]}
    try:
        spark.sql(_PROGRESS_INSERT.format(table=PROGRESS_TABLE), args=args)
    except Exception as e:
        if not _progress_warned:
            reason = str(e).strip().split("\n")[0][:200] or type(e).__name__
            print(f"Progress not saved: {reason}")
            _progress_warned = True

## Summary

- `record()` appends one row per check to `PROGRESS_TABLE`; the latest row per user, workshop and step is the state.
- A failed write prints `Progress not saved: <reason>` once and never stops the workshop.
- `_result()` keeps the PASS / FAIL messages the same in every workshop.

**Next:** the workshop checks `ws_delta` and `ws_ingestion` load this notebook with `%run ./_progress`.